# Echo state networks and autonomous forecasts

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/time-series/echo_state_network.ipynb)

This notebook runs the complete CPU experiment. Results depend on package versions and hardware. No GPU or credentials are required.

In [ ]:
%pip -q install "numpy>=2,<3" "matplotlib>=3.9,<4"

## Sequence and evaluation

A deterministic, discrete Mackey-Glass-style recurrence generates 3000 observations after a 1000-step burn-in. This is a synthetic teaching sequence, not the continuous-time differential equation or a real financial dataset. Standardisation uses the first 1800 observations. Training, validation and test occur in chronological order.

At time t, the reservoir consumes observation t and predicts t+1. A one-step test consumes each actual observation. An autonomous rollout consumes its own preceding prediction. Both start with the reservoir state warmed by the history available at the forecast origin.

In [ ]:
import os, tempfile, time, json
for name in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS']:
    os.environ[name] = '4'
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path(tempfile.mkdtemp(prefix='ml-tutorial-'))
(ROOT / "esn_results").mkdir()
def series(n=3000):
    z=np.full(n+1001,1.2)
    for t in range(18,len(z)-1):z[t+1]=z[t]+.2*z[t-17]/(1+z[t-17]**10)-.1*z[t]
    return z[1000:1000+n]
def reservoir(u,radius,seed,size=80):
    rng=np.random.default_rng(seed);W=rng.normal(size=(size,size))*(rng.random((size,size))<.1)
    W*=radius/np.max(abs(np.linalg.eigvals(W)));win=rng.uniform(-.5,.5,(size,2));state=np.zeros(size);states=[]
    for val in u:
        state=.5*state+.5*np.tanh(W@state+win@np.array([1,val]));states.append(state.copy())
    return np.array(states),W,win
def fit_ridge(A,b,alpha):
    P=np.eye(A.shape[1]);P[0,0]=0
    return np.linalg.solve(A.T@A+alpha*P,A.T@b)
def esn():
    start=time.perf_counter();z=series();mu=z[:1800].mean();sd=z[:1800].std();u=(z-mu)/sd;results=[];art={};radii=[.3,.8,1.2];alphas=[1e-6,1e-4,.01,1.]
    # Features at t predict the observation at t+1.
    for radius in radii:
      for seed in [11,22,33]:
        S,W,win=reservoir(u,radius,seed);A=np.column_stack([np.ones(len(u)),S]);target=u[1:];train=np.arange(100,1799);valid=np.arange(1800,2399);test=np.arange(2400,2999)
        losses=[]
        for alpha in alphas:
            w=fit_ridge(A[train],target[train],alpha);losses.append(np.mean((A[valid]@w-target[valid])**2))
        alpha=alphas[np.argmin(losses)];fit=np.arange(100,2399);w=fit_ridge(A[fit],target[fit],alpha);one=A[test]@w
        state=S[2400].copy();value=u[2400];roll=[]
        for _ in test:
            value=float(np.r_[1,state]@w);roll.append(value)
            state=.5*state+.5*np.tanh(W@state+win@np.array([1,value]))
        truth=target[test];one_rmse=np.sqrt(np.mean((one-truth)**2));roll_rmse=np.sqrt(np.mean((np.array(roll)-truth)**2))
        results.append({'radius':radius,'seed':seed,'alpha':alpha,'validation_mse':float(min(losses)),'one_step_nrmse':float(one_rmse),'rollout_nrmse':float(roll_rmse)})
        if radius==.8 and seed==11:art={'states':S,'one':one*sd+mu,'rollout':np.array(roll)*sd+mu,'truth':truth*sd+mu,'test':test+1,'series':z}
    # Same chronological train/validation/test structure, with 20 lag features.
    lagA=np.array([np.r_[1,u[t-19:t+1][::-1]] for t in range(19,2999)])
    times=np.arange(19,2999);yt=u[times+1];tr=times<1799;va=(times>=1800)&(times<2399);te=times>=2400
    losses=[np.mean((lagA[va]@fit_ridge(lagA[tr],yt[tr],a)-yt[va])**2) for a in alphas];alpha=alphas[np.argmin(losses)];w=fit_ridge(lagA[times<2399],yt[times<2399],alpha)
    one=lagA[te]@w;hist=list(u[2381:2401]);roll=[]
    for _ in range(sum(te)):
        val=float(np.r_[1,np.array(hist[-20:])[::-1]]@w);roll.append(val);hist.append(val)
    baseline={'alpha':alpha,'one_step_nrmse':float(np.sqrt(np.mean((one-yt[te])**2))),'rollout_nrmse':float(np.sqrt(np.mean((roll-yt[te])**2)))}
    art['lag_one']=one*sd+mu;art['lag_rollout']=np.array(roll)*sd+mu
    report={'n':len(z),'reservoir_size':80,'leak_rate':.5,'normalisation_mean':float(mu),'normalisation_std':float(sd),'radii':radii,'alphas':alphas,'results':results,'lagged_ridge':baseline,'elapsed_seconds':time.perf_counter()-start}
    out=ROOT/'esn_results';(out/'results.json').write_text(json.dumps(report,indent=2));np.savez(out/'arrays.npz',**art)
    return report


In [ ]:
report = esn()
for row in report['results']:
    print(row['radius'], row['seed'], round(row['one_step_nrmse'], 5), round(row['rollout_nrmse'], 3))
print('Lagged ridge:', report['lagged_ridge'])
print('Runtime:', report['elapsed_seconds'])

## Forecasts from the same readout

The first panel receives observations throughout the test. The second receives one observed starting value and then feeds predictions back. Large rollout errors are part of the measured result. The linear readout is unconstrained even though tanh bounds individual reservoir states.

In [ ]:
a = np.load(ROOT / 'esn_results/arrays.npz')
fig, axes = plt.subplots(2, 1, figsize=(9,6), sharex=True)
t = np.arange(1,600)
axes[0].plot(t, a['truth'], label='Observed')
axes[0].plot(t, a['one'], '--', label='ESN one step')
axes[1].plot(t, a['truth'], label='Observed')
axes[1].plot(t, a['rollout'], label='ESN autonomous')
axes[1].plot(t, a['lag_rollout'], label='Lagged ridge autonomous')
for ax in axes: ax.legend(); ax.set_ylabel('Series value')
axes[1].set_xlabel('Steps after last observed value')
plt.tight_layout(); plt.show()

## Radius and regularisation

Only the readout is fitted. Three reservoir radii and three seeds make nine configurations. Four ridge penalties are selected using one-step validation error. This selection criterion is deliberately narrower than long-horizon rollout error. Recurrent spectral radius alone does not certify stability of the closed-loop reservoir plus readout.

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(10,4))
for seed in [11,22,33]:
    rows = [r for r in report['results'] if r['seed']==seed]
    for ax, metric in zip(axes, ['one_step_nrmse','rollout_nrmse']):
        ax.plot([r['radius'] for r in rows], [r[metric] for r in rows], 'o-', label=f'Seed {seed}')
        ax.set(xlabel='Spectral radius', ylabel=metric); ax.legend()
axes[1].set_yscale('log'); plt.tight_layout(); plt.show()

## Exercises

1. Select ridge regularisation using autonomous validation rollouts and evaluate on the untouched test period.
2. Remove the leaky update and compare state dynamics.
3. Repeat with observational noise and report error against both noisy observations and clean latent values.
4. Try different washout lengths while keeping chronological boundaries fixed.

## References

- [Jaeger (2001), The echo state approach, corrected 2010 version](https://www.ai.rug.nl/minds/uploads/EchoStatesTechRep.pdf).
- [ReservoirPy quick start](https://reservoirpy.readthedocs.io/en/latest/user_guide/quickstart.html).

## Tested environment
Executed on Python 3.12 with NumPy 2.5.3, SciPy 1.18.1, scikit-learn 1.9.1 and Matplotlib 3.11.2. Ranking additionally used XGBoost 3.4.1. Package versions and floating-point libraries can affect timings and fitted results.
